In [4]:
import math
from apeGmsh import apeGmsh

a, L, twist, N = 1.0, 10, math.radians(90), 20

g = apeGmsh(model_name="twisted_prism")
g.begin()

wires = []
for i in range(N + 1):
    y = (i / N) * L
    c, s = math.cos((i / N) * twist), math.sin((i / N) * twist)
    pts = [g.model.geometry.add_point(c*lx + s*lz, y, -s*lx + c*lz)
           for lx, lz in [(-a,-a), (a,-a), (a,a), (-a,a)]]
    lines = [g.model.geometry.add_line(pts[i], pts[(i+1) % 4]) for i in range(4)]
    wires.append(g.model.geometry.add_wire(lines, check_closed=True))

out = g.model.transforms.thru_sections(wires, make_solid=True)
volume = next(t for d, t in out if d == 3)

g.physical.add(3, [volume], name="Body")

g.model.viewer()

g.mesh.sizing.set_global_size(max_size=0.25, min_size=0.25)
g.mesh.generation.generate(dim=3)
g.mesh.viewer()